# Antmaze — MCTS variants (slow) + upper-bound band

Pha CHẬM: MCTS (E1a: +suffix/pw/bayes; E1c: nofb/fb) vs soft_floyd, **paired**. E1c dùng `--noise-seeds 0 1 2` (band) + σ tới 0.5 để định vị **σ\*** (gap MCTS−Floyd đỉnh, CI tách 0) và **σ_all_fail** (mọi planner sập). ~1–2h — tách riêng để không kéo phần classical. Chạy `antmaze_classical` trước.

## 1. Code + env (~10–15 phút lần đầu)

In [ ]:
import os
if os.path.isdir('/kaggle/working/latent_landmarks'):
    !cd /kaggle/working/latent_landmarks && git pull -q origin retrain
else:
    !git clone -q -b retrain https://github.com/Jun1801/latent_landmarks.git /kaggle/working/latent_landmarks
if not os.path.isdir('/kaggle/working/wmag'):
    !git clone -q https://github.com/LunjunZhang/world-model-as-a-graph /kaggle/working/wmag
!bash /kaggle/working/latent_landmarks/repro/setup_kaggle.sh

## 2. Restore checkpoint `antmaze_s221_paper` (set `SLUG`)

In [ ]:
!ls /kaggle/input/
import os, shutil
SLUG = 'PUT-DATASET-SLUG-HERE'          # <-- sửa cho khớp /kaggle/input/
CKPT, ENV = 'antmaze_s221_paper', 'AntMaze-v1'
src = f'/kaggle/input/{SLUG}/experiments/{ENV}/{CKPT}/state'
dst = f'/kaggle/working/experiments/{ENV}/{CKPT}/state'; os.makedirs(dst, exist_ok=True)
for f in ['agent.pt', 'algo.pt']:
    shutil.copy(f'{src}/{f}', f'{dst}/{f}')
print('restored ->', os.listdir(dst))

## 3. Verify GPU + env

In [ ]:
!export PATH=/opt/conda/bin:$PATH; export LD_LIBRARY_PATH=$HOME/.mujoco/mujoco200/bin:/usr/lib/nvidia:${LD_LIBRARY_PATH:-};  conda run -n l3p python -c "import torch,mujoco_py; print('cuda',torch.cuda.is_available())"

## 4. Sanity σ=0 (soft_floyd/mcts_nofb/mcts_fb trùng nhau)

In [ ]:
!export PATH=/opt/conda/bin:$PATH; export LD_LIBRARY_PATH=$HOME/.mujoco/mujoco200/bin:/usr/lib/nvidia:${LD_LIBRARY_PATH:-};  conda run -n l3p python /kaggle/working/latent_landmarks/repro/paper_mcts/eval_ablation.py --env antmaze --resume_ckpt antmaze_s221_paper --sims 100 --regime e1c --episodes 2 --n_test_rollouts 20 --planners soft_floyd mcts_nofb mcts_fb --sigmas 0

## 5. E1a (stochastic) — MCTS variants

In [ ]:
!export PATH=/opt/conda/bin:$PATH; export LD_LIBRARY_PATH=$HOME/.mujoco/mujoco200/bin:/usr/lib/nvidia:${LD_LIBRARY_PATH:-};  conda run -n l3p python /kaggle/working/latent_landmarks/repro/paper_mcts/eval_ablation.py --env antmaze --resume_ckpt antmaze_s221_paper --sims 100 --regime e1a --episodes 3 --n_test_rollouts 30 --planners soft_floyd mcts mcts+suffix mcts+pw mcts+bayes --out /kaggle/working/exp_out/antmaze_e1a_mcts.json --sigmas 0 5 10 20

## 6. E1c (bias) — MCTS band, σ tới 0.5 (upper bound)  ⏳ long pole ~30–60′

In [ ]:
!export PATH=/opt/conda/bin:$PATH; export LD_LIBRARY_PATH=$HOME/.mujoco/mujoco200/bin:/usr/lib/nvidia:${LD_LIBRARY_PATH:-};  conda run -n l3p python /kaggle/working/latent_landmarks/repro/paper_mcts/eval_ablation.py --env antmaze --resume_ckpt antmaze_s221_paper --sims 100 --regime e1c --episodes 3 --n_test_rollouts 30 --planners soft_floyd mcts_nofb mcts_fb --noise-seeds 0 1 2 --out /kaggle/working/exp_out/antmaze_e1c_mcts.json --sigmas 0 0.05 0.1 0.15 0.2 0.25 0.3 0.4 0.5

## 7. Dump 3-planner routes (hình plan-comparison / wormhole), 1 episode

In [ ]:
!export PATH=/opt/conda/bin:$PATH; export LD_LIBRARY_PATH=$HOME/.mujoco/mujoco200/bin:/usr/lib/nvidia:${LD_LIBRARY_PATH:-};  conda run -n l3p python /kaggle/working/latent_landmarks/repro/paper_mcts/eval_ablation.py --env antmaze --resume_ckpt antmaze_s221_paper --sims 100 --regime e1c --episodes 1 --n_test_rollouts 1 --sigmas 0.2 --dump-plans /kaggle/working/exp_out/antmaze_plans_e1c.json

## 8. Lấy JSON về
Tải `antmaze_e1a_mcts.json`, `antmaze_e1c_mcts.json`, `antmaze_plans_e1c.json` → gửi lại.

In [ ]:
!ls -la /kaggle/working/exp_out/ 2>/dev/null || echo 'chưa có output'